In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

print("Environment ready")

Environment ready


In [3]:
train = pd.read_csv("../Data/train.csv")
test = pd.read_csv("../Data/test.csv")

print("Train:", train.shape)
print("Test :", test.shape)

Train: (668665, 15)
Test : (286571, 14)


In [4]:
train.head()
train.info()
train.describe(include="all").T

<class 'pandas.DataFrame'>
RangeIndex: 668665 entries, 0 to 668664
Data columns (total 15 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   id                           668665 non-null  int64  
 1   Age                          668665 non-null  int64  
 2   Annual_Income_USD            668665 non-null  float64
 3   Daily_Commute_km             668665 non-null  float64
 4   Number_of_Cars_Owned         668665 non-null  int64  
 5   Charging_Stations_Near_Home  668665 non-null  int64  
 6   Charging_Stations_Near_Work  668665 non-null  int64  
 7   Environmental_Concern_Level  668665 non-null  float64
 8   Gender                       668665 non-null  str    
 9   City_Type                    668665 non-null  str    
 10  Current_Car_Type             668665 non-null  str    
 11  Home_Charging_Possible       668665 non-null  str    
 12  Subsidy_Available            668665 non-null  str    
 13  Range_Anxi

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
id,668665.0,NaN,NaN,NaN,334332.0,193027.103211,0.0,167166.0,334332.0,501498.0,668664.0
Age,668665.0,NaN,NaN,NaN,47.039171,12.875448,25.0,36.0,47.0,58.0,69.0
Annual_Income_USD,668665.0,NaN,NaN,NaN,84769.266989,28648.029042,30000.0,67376.0,84880.0,102753.0,188549.0
Daily_Commute_km,668665.0,NaN,NaN,NaN,32.158298,18.730474,5.0,17.2,33.6,47.4,98.7
Number_of_Cars_Owned,668665.0,NaN,NaN,NaN,1.712626,0.729275,1.0,1.0,2.0,2.0,4.0
Charging_Stations_Near_Home,668665.0,NaN,NaN,NaN,4.960408,3.926843,0.0,2.0,4.0,7.0,14.0
Charging_Stations_Near_Work,668665.0,NaN,NaN,NaN,7.176314,5.186627,0.0,3.0,6.0,10.0,19.0
Environmental_Concern_Level,668665.0,NaN,NaN,NaN,2.935477,1.429119,1.0,2.0,3.0,4.0,5.0
Gender,668665,3,Male,367954,NaN,NaN,NaN,NaN,NaN,NaN,NaN
City_Type,668665,3,Urban,289305,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
TARGET = "Will_Buy_EV"

print(train[TARGET].value_counts())
print()
print(train[TARGET].value_counts(normalize=True))

Will_Buy_EV
No     551886
Yes    116779
Name: count, dtype: int64

Will_Buy_EV
No     0.825355
Yes    0.174645
Name: proportion, dtype: float64


In [6]:
missing = train.isna().sum()

missing[missing > 0].sort_values(ascending=False)

print("Duplicate rows:", train.duplicated().sum())

Duplicate rows: 0


In [7]:
X = train.drop(columns=[TARGET])
y = train[TARGET]

categorical_cols = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numerical_cols = X.select_dtypes(
    include=np.number
).columns.tolist()

print("Categorical columns:")
print(categorical_cols)

print("\nNumerical columns:")
print(numerical_cols)

Categorical columns:
['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']

Numerical columns:
['id', 'Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned', 'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work', 'Environmental_Concern_Level']


/tmp/ipykernel_6081/4162106341.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(


In [8]:
from sklearn.model_selection import train_test_split

# Remove ID from the features
X = train.drop(columns=[TARGET, "id"])
y = train[TARGET]

# Convert target to 0/1 if necessary
if y.dtype == "object":
    print("Target values:", y.unique())

In [9]:
print("Target dtype:", y.dtype)
print("Target values:")
print(y.value_counts())

Target dtype: str
Target values:
Will_Buy_EV
No     551886
Yes    116779
Name: count, dtype: int64


In [10]:
y = y.map({"No": 0, "Yes": 1})

print(y.dtype)
print(y.value_counts())

int64
Will_Buy_EV
0    551886
1    116779
Name: count, dtype: int64


In [11]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training:", X_train.shape)
print("Validation:", X_valid.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True))

print("\nValidation target distribution:")
print(y_valid.value_counts(normalize=True))

Training: (534932, 13)
Validation: (133733, 13)

Training target distribution:
Will_Buy_EV
0    0.825355
1    0.174645
Name: proportion, dtype: float64

Validation target distribution:
Will_Buy_EV
0    0.825354
1    0.174646
Name: proportion, dtype: float64


In [12]:
categorical_cols = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Categorical columns:")
print(categorical_cols)

Categorical columns:
['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']


/tmp/ipykernel_6081/3802955034.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(


In [13]:
from catboost import CatBoostClassifier

catboost_model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=7,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=100
)

catboost_model.fit(
    X_train,
    y_train,
    cat_features=categorical_cols,
    eval_set=(X_valid, y_valid),
    early_stopping_rounds=50
)

0:	test: 0.9322243	best: 0.9322243 (0)	total: 254ms	remaining: 2m 6s
100:	test: 0.9396858	best: 0.9396858 (100)	total: 11.5s	remaining: 45.5s
200:	test: 0.9402720	best: 0.9402720 (200)	total: 22.3s	remaining: 33.2s
300:	test: 0.9406329	best: 0.9406329 (300)	total: 33.3s	remaining: 22s
400:	test: 0.9409926	best: 0.9409927 (399)	total: 45s	remaining: 11.1s
499:	test: 0.9411934	best: 0.9411934 (499)	total: 56.7s	remaining: 0us

bestTest = 0.9411933765
bestIteration = 499



CatBoostClassifier(depth=7, eval_metric='AUC', iterations=500, learning_rate=0.05, loss_function='Logloss', random_seed=42, verbose=100)

In [14]:
valid_pred = catboost_model.predict_proba(X_valid)[:, 1]

auc = roc_auc_score(y_valid, valid_pred)

print(f"Validation ROC-AUC: {auc:.6f}")

Validation ROC-AUC: 0.941193


## Experiment 001 - CatBoost Baseline

- Features: Original features, ID removed
- Train/validation split: 80/20
- Stratification: Yes
- Random seed: 42
- Model: CatBoostClassifier
- Iterations: 500
- Learning rate: 0.05
- Depth: 7
- Metric: ROC-AUC

**Validation ROC-AUC: 0.941193**

In [15]:
# Full training data
X_full = train.drop(columns=[TARGET, "id"])
y_full = train[TARGET]

# Test data
X_test = test.drop(columns=["id"])

# Categorical columns
categorical_cols = X_full.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Training shape:", X_full.shape)
print("Test shape:", X_test.shape)
print("Categorical:", categorical_cols)

Training shape: (668665, 13)
Test shape: (286571, 13)
Categorical: ['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']


/tmp/ipykernel_6081/3187341127.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_full.select_dtypes(


In [16]:
final_catboost = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=7,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=100
)

final_catboost.fit(
    X_full,
    y_full,
    cat_features=categorical_cols
)

0:	total: 218ms	remaining: 1m 48s
100:	total: 13.4s	remaining: 53s
200:	total: 25.7s	remaining: 38.3s
300:	total: 38.4s	remaining: 25.4s
400:	total: 51.7s	remaining: 12.8s
499:	total: 1m 5s	remaining: 0us


CatBoostClassifier(depth=7, eval_metric='AUC', iterations=500, learning_rate=0.05, loss_function='Logloss', random_seed=42, verbose=100)

In [17]:
test_pred = final_catboost.predict_proba(X_test)[:, 1]

print(test_pred[:10])
print("Min:", test_pred.min())
print("Max:", test_pred.max())

[8.69632874e-03 2.22507495e-02 4.99588464e-03 2.07944442e-03
 2.34764023e-02 3.65702698e-03 3.85558970e-01 1.19171589e-04
 6.55026196e-01 6.46028024e-03]
Min: 1.1596377615855225e-06
Max: 0.9762150339995113


In [18]:
submission = pd.DataFrame({
    "id": test["id"],
    TARGET: test_pred
})

submission.head()

print("Submission shape:", submission.shape)
print("Test shape:", test.shape)

submission.isna().sum()

Submission shape: (286571, 2)
Test shape: (286571, 14)


id             0
Will_Buy_EV    0
dtype: int64

In [19]:
import os
os.makedirs("../submissions", exist_ok=True)

submission.to_csv("../submissions/catboost_baseline.csv", index=False)